# GazeformerISP-S: PerGAZE to COCO-FV transfer on Colab

Train only new COCO-FV subject embedding rows. All other parameters, old subject rows and BatchNorm buffers are frozen; dropout stays disabled.

Select a GPU runtime. Inputs and outputs are read/written directly on Drive.
K=5, seed=10 chooses 50 support records from the 6,462 COCO-FV test records; the
other 6,412 records are evaluated. The split matches ChenLSTM transfer for the
same input/K/seed. Different subjects on the same image can occur on both sides.

Free-viewing is represented by a fixed 768D sentence vector from
sentence-transformers/stsb-roberta-base-v2. The notebook creates a separate NPZ
once. No task-head selection, external attention maps or image feature cache.

The remaining test selects the best epoch by harmonic mean of both ScanMatch
metrics; this is model-selected performance, not an untouched test estimate.
Keep separate runs for predictor FT and embedding-only ISP-S.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import subprocess
import sys
import shutil
from pathlib import Path

REPO_URL = "https://github.com/ntnghia06/IndividualScanpath.git"
CODE = Path("/content/IndividualScanpath")
# Optional Drive override when the new code has not been pushed to GitHub.
# Upload both the selected branch's src/ and gazeformer_transfer/ under this folder.
TRANSFER_SOURCE = Path("/content/drive/MyDrive/PerGAZE/transfer_code/COCO-FV")
if (CODE / ".git").is_dir():
    subprocess.run(["git", "-C", str(CODE), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CODE)], check=True)
TRANSFER_ROOT = CODE / "PerGAZE/Transfer/COCO-FV"
if TRANSFER_SOURCE is not None and TRANSFER_SOURCE.is_dir():
    for relative in ("gazeformer_transfer", "GazeformerISP-S/src"):
        source, destination = TRANSFER_SOURCE / relative, TRANSFER_ROOT / relative
        assert source.is_dir(), f"Missing Drive source: {source}"
        if source.resolve() != destination.resolve():
            shutil.copytree(source, destination, dirs_exist_ok=True,
                            ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "tests", "runs"))
SRC = TRANSFER_ROOT / "GazeformerISP-S/src"
for path in (SRC / "train.py", SRC / "test.py", SRC / "prepare_text.py",
             TRANSFER_ROOT / "gazeformer_transfer/engine.py",
             CODE / "PerGAZE/GazeformerISP/src/models/gazeformer/model.py"):
    assert path.is_file(), (f"Missing {path}. Push the new code to GitHub, or upload "
        "the selected branch and gazeformer_transfer to TRANSFER_SOURCE on Drive.")
print("Source:", SRC)


In [ ]:
%pip install -q "numpy>=1.24" "Pillow>=9.1" "scipy>=1.10" "tqdm>=4.65" multimatch-gaze scikit-image "sentence-transformers>=2.2"


In [ ]:
import torch

assert torch.cuda.is_available(), "Select GPU in Runtime > Change runtime type"
GPU_IDS = [0]
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory GiB:", torch.cuda.get_device_properties(0).total_memory / 2**30)


## Drive paths and settings

Default input: MyDrive/PerGAZE/coco_freeview with COCOFreeView_fixations_test.json and images/.
BASE_RUN defaults to MyDrive/PerGAZE/pergaze_gazeformer_run_online. Set CHECKPOINT
to the actual trained PerGAZE geometry-v3 model, not a functional-test checkpoint.
The free-viewing NPZ is created at TEXT using the same text encoder as base training;
the source PerGAZE task NPZ is not required or changed. Resume verifies its hash.

The full source repository is needed. If the new transfer source is not on GitHub,
upload Transfer/gazeformer_transfer and the selected Transfer/GazeformerISP-FT or
Transfer/GazeformerISP-S branch under TRANSFER_SOURCE from the setup cell.

Images: ToTensor -> Resize(768,1024) -> Normalize. The checkpoint's scanpath frame
is 512x352 with 24x32 grid (769 actions). Prediction X/Y are exported in original
COCO-FV image pixels, T in milliseconds. No DINOv2 or saved image cache is used.

Set RESUME=True after a disconnect, keeping the same source checkpoint, input JSON,
free-viewing NPZ, K, seed, prompt and optimization settings. A head-only checkpoint
needs internet once to initialize the same COCO backbone; online checkpoints carry it.


In [ ]:
import json
import math
from collections import Counter

K = 5
SEED = 10
EVAL_SEED = 10
EPOCHS = 3
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
GRAD_CLIP = 12.5
TRAIN_BATCH = 1
TEST_BATCH = 1
WORKERS = 2
INIT = "mean"  # Random uses std=1.0, matching GazeformerISP-S.
EVAL_REPEAT_NUM = 1
GREEDY = False
MAX_BATCHES = 0       # 0=all support; 1=one-batch functional check
EVAL_MAX_BATCHES = 0  # 0=all remaining test; 1=one-batch functional check
RESUME = False

DRIVE_ROOT = Path("/content/drive/MyDrive/PerGAZE")
DATA = DRIVE_ROOT / "coco_freeview"
TEST_FILE = DATA / "COCOFreeView_fixations_test.json"
IMAGE_DIR = DATA / "images"
TEXT = DATA / "gazeformer_freeview_embeddings.npz"
PROMPT = "You are allowed to view visual stimuli or scenes without any specific task or instruction."
BASE_RUN = DRIVE_ROOT / "pergaze_gazeformer_run_online"
CHECKPOINT = BASE_RUN / "checkpoints/best.pth"
RUN = DRIVE_ROOT / "runs" / f"cocofv_gazeformer_s_k{K}_seed{SEED}_colab_1gpu"
LOCAL_ROOT = Path("/content/cocofv_gazeformer_s_work")  # Subprocess logs / optional ZIP only.

assert TEST_FILE.is_file(), f"Missing COCO-FV test JSON: {TEST_FILE}"
assert IMAGE_DIR.is_dir(), f"Missing COCO-FV images: {IMAGE_DIR}"
assert CHECKPOINT.is_file(), f"Missing trained PerGAZE Gazeformer checkpoint: {CHECKPOINT}"
assert min(K, EPOCHS, TRAIN_BATCH, TEST_BATCH, EVAL_REPEAT_NUM) >= 1
assert math.isfinite(LEARNING_RATE) and LEARNING_RATE > 0
assert math.isfinite(WEIGHT_DECAY) and WEIGHT_DECAY >= 0
assert math.isfinite(GRAD_CLIP) and min(WORKERS, MAX_BATCHES, EVAL_MAX_BATCHES) >= 0
assert INIT in ("mean", "random") and PROMPT.strip()
assert 0 <= SEED < 2**32 and 0 <= EVAL_SEED < 2**32
assert WEIGHT_DECAY == 0, "ISP-S requires zero weight decay"
rows = json.loads(TEST_FILE.read_text(encoding="utf-8-sig"))
assert rows and all(row.get("split") == "test" for row in rows), "Use the COCO-FV test JSON only"
counts = Counter(row["subject"] for row in rows)
assert all(count > K for count in counts.values()), "K must leave evaluation records for every subject"
missing = sorted({row["name"] for row in rows if not (IMAGE_DIR / row["name"]).is_file()})
assert not missing, f"Missing {len(missing)} COCO-FV images; first: {missing[:5]}"
RUN.mkdir(parents=True, exist_ok=True)
last_checkpoint = RUN / "checkpoints/checkpoint.pth"
if RESUME:
    assert last_checkpoint.is_file(), f"No transfer checkpoint at {last_checkpoint}"
    assert TEXT.is_file(), "Resume requires the original free-viewing NPZ"
else:
    assert not last_checkpoint.exists(), "Existing run: set RESUME=True or choose a new RUN"
print("COCO-FV observers:", len(counts), "source test samples:", len(rows))
print("Selected support:", K * len(counts), "remaining evaluation:", len(rows) - K * len(counts))
print("Mode: embedding; image backbone frozen; prompt:", PROMPT)
print("Drive input:", DATA)
print("PerGAZE base checkpoint:", CHECKPOINT)
print("Drive output:", RUN)


In [ ]:
import shutil

COMMON = [
    "--checkpoint", str(CHECKPOINT),
    "--data_root", str(DATA), "--test_file", str(TEST_FILE), "--img_dir", str(IMAGE_DIR),
    "--text_embeddings", str(TEXT), "--prompt", PROMPT,
    "--k", str(K), "--seed", str(SEED), "--eval_seed", str(EVAL_SEED),
    "--epoch", str(EPOCHS), "--lr", str(LEARNING_RATE),
    "--weight_decay", str(WEIGHT_DECAY), "--clip", str(GRAD_CLIP), "--init", INIT,
    "--eval_repeat_num", str(EVAL_REPEAT_NUM), "--max_batches", str(MAX_BATCHES),
    "--eval_max_batches", str(EVAL_MAX_BATCHES), "--device", "cuda:0", "--gpu_ids", "0",
]
if GREEDY:
    COMMON.append("--greedy")

def run_script(script, *arguments, use_common=True):
    import codecs
    import time

    command = [sys.executable, "-u", str(SRC / script), *(COMMON if use_common else []), *arguments]
    local_logs = LOCAL_ROOT / "logs"
    local_logs.mkdir(parents=True, exist_ok=True)
    log_path = local_logs / f"{Path(script).stem}_{time.time_ns()}.log"
    drive_log = RUN / "logs" / log_path.name
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    tail = bytearray()
    process = None
    return_code = None
    print("Full subprocess log:", log_path, flush=True)
    try:
        with log_path.open("wb") as log:
            log.write(("Command: " + repr(command) + "\n").encode("utf-8"))
            process = subprocess.Popen(command, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, cwd=str(CODE))
            try:
                while True:
                    chunk = process.stdout.read1(8192)
                    if not chunk:
                        break
                    log.write(chunk)
                    log.flush()
                    tail.extend(chunk)
                    if len(tail) > 65536:
                        del tail[:-65536]
                    sys.stdout.write(decoder.decode(chunk))
                    sys.stdout.flush()
                sys.stdout.write(decoder.decode(b"", final=True))
                return_code = process.wait()
            finally:
                if process.poll() is None:
                    process.terminate()
                    try:
                        process.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        process.wait()
                process.stdout.close()
    finally:
        if log_path.is_file():
            try:
                drive_log.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(log_path, drive_log)
                print("Log copied to Drive:", drive_log, flush=True)
            except OSError as error:
                print("Could not copy log to Drive; local log is retained:", error, flush=True)
    if return_code:
        lines = tail.decode("utf-8", errors="replace").replace("\r", "\n").splitlines()
        last_lines = "\n".join(lines[-40:])
        hint = ""
        if "out of memory" in last_lines.lower():
            hint = "\nMemory error: for a NEW run try TRAIN_BATCH=1, TEST_BATCH=1. Resume requires the saved train batch."
        elif "checkpoint exists" in last_lines.lower():
            hint = "\nSet RESUME=True to continue that run, or choose a new RUN directory."
        raise RuntimeError(f"{script} exited with code {return_code}. Full log: {log_path}"
                           f"\nLast subprocess output:\n{last_lines}{hint}")


## Prepare support/text, then adapt

The split preparation validates the COCO-FV input and saves support/remainder without
loading weights. Text preparation creates the 768D free-viewing NPZ once or validates
and reuses it; the text encoder runs in a separate process and is not finetuned.
Training evaluates a baseline, adapts on support and evaluates remaining test after
each epoch. The frozen image backbone runs online with no image feature cache.

Last/best checkpoints, optimizer, RNG, report and predictions persist on Drive.
Full subprocess logs and original tracebacks are copied to RUN/logs. Batch limits
are for functional checks; reports flag incomplete training/evaluation.


In [ ]:
if not RESUME:
    run_script("train.py", "--prepare_only", "--log_root", str(RUN), "--workers", "0")
    selection = json.loads((RUN / "selection.json").read_text(encoding="utf-8"))
    print("Support:", selection["support_samples"], "remaining:", selection["remaining_test_samples"])
run_script("prepare_text.py", "--output", str(TEXT), "--prompt", PROMPT,
           "--device", "cuda:0", use_common=False)
arguments = ["--log_root", str(RUN), "--workers", str(WORKERS),
             "--batch", str(TRAIN_BATCH), "--test_batch", str(TEST_BATCH)]
if RESUME:
    arguments.append("--resume")
run_script("train.py", *arguments)


In [ ]:
report = json.loads((RUN / "report.json").read_text(encoding="utf-8"))
print("Mode:", report["mode"], "frozen image backbone:", report["frozen_backbone"])
print("Support:", report["support_samples"], "remaining:", report["test_samples"])
print("COCO-FV subjects:", report["adaptive_subjects"])
print("Prompt:", report["prompt"])
print("Trainable:", report["trainable_parameter"], "elements:", report["trainable_elements"])
print("Best epoch:", report["best_epoch"], "score:", report["best_score"])
print(json.dumps(report["best_metrics"]["overall"], indent=2))
print(json.dumps(report["best_metrics"].get("retrieval", {}).get("by_condition", {}), indent=2))
print("Report:", RUN / "report.json")


## Optional reevaluation

Uncomment the next cell to reevaluate the adapted best.pth on the same remaining
test. The last --checkpoint overrides the base path in COMMON. Original test JSON
and free-viewing NPZ hashes are verified; selected support remains excluded.
Results append to report.json and predictions_reevaluation.json.


In [ ]:
# run_script("test.py", "--checkpoint", str(RUN / "checkpoints/best.pth"),
#            "--log_root", str(RUN), "--workers", str(WORKERS),
#            "--test_batch", str(TEST_BATCH))


## Resume and outputs

Mount Drive and rerun setup with RESUME=True, the same RUN, original source
checkpoint, test JSON, free-viewing NPZ and adaptation settings. Predictor FT and
embedding-only ISP-S checkpoints cannot resume/evaluate as the other mode.
A partial epoch restarts from the last completed checkpoint.

Outputs: support.json, test_remaining.json, selection.json, checkpoints/checkpoint.pth,
checkpoints/best.pth, report.json, predictions_best.json, predictions_last.json and
logs/. X/Y use original image pixels, T uses milliseconds; scanpath retains 512x352
model coordinates and seconds for metrics. Keep the free-viewing NPZ with run artifacts;
FT and ISP-S can share the same NPZ. Optional ZIP download is below.


In [ ]:
print("Outputs saved on Drive:", RUN)
# LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
# archive = shutil.make_archive(str(LOCAL_ROOT / f"cocofv_gazeformer_s_k{K}_seed{SEED}"),
#                              "zip", root_dir=RUN.parent, base_dir=RUN.name)
# from google.colab import files
# files.download(archive)
